# REPAIR参数校准前的数据审计

## tl;dr
现有记录可以核对真实执行及会计；无法据此声称长挂单参数已校准或策略整体正EV。固定NORMAL的12场与47场Shadow没有重合，19场旧日志可对齐但策略不同。

## Context & Methods
这是诊断配套Notebook，不是盈利回测。时区Asia/Shanghai，来源及哈希见evidence.json。

### Key Assumptions
保持固定原开仓事实；无费用。最终赢家未知时只展示经济结果区间。客户端观测时间与交易所撮合时间分别处理；as-of使用实际可用时间。旧组不并入固定NORMAL收益验证。


## Data
### 1. 重读原始文件
只读原始journal及快照；重写同目录的审计CSV/JSON。冻结12场原始文件校验既有SHA256，其他文件记录本次哈希。

In [1]:
from pathlib import Path
import contextlib, csv, io, json, math, runpy
relative = Path('docs/development/reviews/20261018_repair_ev_calibration')
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / relative / 'audit.py').exists())
out = root / relative
with contextlib.redirect_stdout(io.StringIO()):
    evidence = runpy.run_path(str(out / 'audit.py'))['main']()
print('Reviewed source files:', len(evidence['sources']))
print('Prior SHA256 checks:', sum(s['prior_hash_checked'] for s in evidence['sources']))
print(json.dumps(evidence['shadow'], ensure_ascii=False, indent=2))


Reviewed source files: 126
Prior SHA256 checks: 12
{
  "sessions": 47,
  "local_journal_matches": 19,
  "fixed_normal_cohort_matches": 0,
  "snapshot_rows": 14100,
  "expected_slots": 14100,
  "valid_probability_rows": 10054,
  "missed_rows": 541,
  "reported_dropped_events": 0,
  "reported_writer_errors": 0,
  "undrained_writers": 0
}


## Results
### 2. 分组保留分母
placement结果包含拒单和未发送记录；真实submitted计数单列。组内订单与同一市场相关，不视作独立市场试验。

In [2]:
for name, group in evidence['cohorts'].items():
    print(json.dumps({'cohort': name, 'markets': group['sessions'],
        'submitted_repair_results': group['submitted_repair_placement_results'],
        'repair_orders_with_confirmed_fill': group['repair_orders_with_confirmed_fill'],
        'confirmed_fills_all_phases': group['confirmed_fills'],
        'repair_first_cancel_request_age_s': group['repair_cancel_request_age_s'],
        'native_match_time_fills': group['positive_exchange_match_time_fills']}, ensure_ascii=False, indent=2))


{
  "cohort": "fixed_normal_20261017",
  "markets": 12,
  "submitted_repair_results": 80,
  "repair_orders_with_confirmed_fill": 14,
  "confirmed_fills_all_phases": 33,
  "repair_first_cancel_request_age_s": {
    "n": 70,
    "min": 0.23109674453735352,
    "median": 1.0393271446228027,
    "p90": 3.003621816635132,
    "max": 3.007023334503174
  },
  "native_match_time_fills": 0
}
{
  "cohort": "older_shadow_joined",
  "markets": 19,
  "submitted_repair_results": 86,
  "repair_orders_with_confirmed_fill": 28,
  "confirmed_fills_all_phases": 100,
  "repair_first_cancel_request_age_s": {
    "n": 66,
    "min": 3.001310348510742,
    "median": 3.0022681951522827,
    "p90": 3.003265857696533,
    "max": 3.004101037979126
  },
  "native_match_time_fills": 0
}


### 3. 核对观测区间与成交身份
保守下界是本地POST起点或旧版提交intent，下界不是上次缓存未成交时间。上界是首次本地MATCHED，缺时用CONFIRMED。

In [3]:
with (out / 'fills.csv').open() as f:
    fills = list(csv.DictReader(f))
keys = [(r['session_id'], r['trade_id'], r['order_id']) for r in fills]
assert len(keys) == len(set(keys))
for row in fills:
    lower, upper = row['conservative_match_lower_bound'], row['conservative_match_upper_bound']
    assert not lower or float(lower) <= float(upper)
    assert not row['asof_snapshot_age_s'] or float(row['asof_snapshot_age_s']) >= 0
assert not evidence['issues']
print('Unique confirmed fills:', len(fills))
print('Trade state sets:', evidence['trade_identity_status_sets'])
print('Valid positive native timestamp events:', evidence['trade_event_timestamps']['positive_native_ts'])


Unique confirmed fills: 133
Trade state sets: {'CONFIRMED|MATCHED|MINED': 121, 'CONFIRMED|MATCHED': 12}
Valid positive native timestamp events: 0


### 4. 最终会计与未知结算
累计经济买入的两种兑付结果分别计算；不把Merge后的物理余额减全部历史成本。图形对这里的身份及恒等式检查无额外帮助，保留精确值表。

In [4]:
with (out / 'terminal_accounting.csv').open() as f:
    accounting = list(csv.DictReader(f))
for row in accounting:
    for outcome in ('yes', 'no'):
        total = float(row['total_pnl_if_' + outcome])
        normal = float(row['normal_only_pnl_if_' + outcome])
        repair = float(row['repair_increment_if_' + outcome])
        assert math.isclose(total, normal + repair, abs_tol=1e-8)
for name, group in evidence['cohorts'].items():
    print(name)
    print(json.dumps(group['terminal_accounting'], ensure_ascii=False, indent=2))
print('Accounting identities checked:', len(accounting) * 2)


fixed_normal_20261017
{
  "markets_with_recorded_winner": 0,
  "all_buy_cost_pnl_floor_usdc": -7.210200000000001,
  "residual_shares": 0.019999999999999574,
  "actual_pnl_outcome_bounds_usdc": [
    -7.210200000000001,
    -7.190200000000002
  ],
  "repair_increment_outcome_bounds_usdc": [
    -28.2602,
    16.7598
  ],
  "max_normal_cost_per_market_usdc": 8.7
}
older_shadow_joined
{
  "markets_with_recorded_winner": 0,
  "all_buy_cost_pnl_floor_usdc": -37.857499000000004,
  "residual_shares": 33.967339,
  "actual_pnl_outcome_bounds_usdc": [
    -37.857499000000004,
    -3.8901599999999994
  ],
  "repair_increment_outcome_bounds_usdc": [
    -71.145408,
    36.088391
  ],
  "max_normal_cost_per_market_usdc": 16.730000000000004
}
Accounting identities checked: 62


## Takeaways
1. 演示中的成交强度、队列和长挂单参数仍未校准，不能根据某个参数给出正收益就选择该参数。
2. 补权威最终赢家后，可用G=sum x*(Z_side-b)学习已有动作的终端收益贡献，无需精确撮合秒数。赢家只用作事后标签。
3. 未执行过的长挂单仍没有反事实标签，需额外结构与实证支持。
4. 固定组历史总PnL约[-7.2102,-7.1902]USDC是本批路径结果，不是EV估计，也不证明每次repair降低EV。

验证说明：本Notebook各代码单元已按顺序在同一标准库Python命名空间执行并保存输出；未使用Jupyter内核，未进行浏览器排版验收。可在Jupyter环境使用nbconvert重新执行。
